In [5]:
# Source: Mistral AI
class LFSRScrambler:
    def __init__(self, seed=0xFF, polynomial=0x8A):
        """
        Initialize the LFSR scrambler with a seed and a polynomial.
        The polynomial 0x8A corresponds to x^8 + x^7 + x^5 + x^3 + 1.
        """
        self.seed = seed
        self.polynomial = polynomial
        self.reset()

    def reset(self):
        """
        Reset the LFSR state to the initial seed.
        """
        self.state = self.seed

    def scrambler(self, data):
        """
        Scramble the input data using the LFSR.
        """
        scrambled_data = []
        for byte in data:
            scrambled_byte = byte ^ self.state
            scrambled_data.append(scrambled_byte)
            # Update the LFSR state
            self.state = self._update_state()
        return bytes(scrambled_data)

    def descrambler(self, data):
        """
        Descramble the input data using the LFSR.
        """
        self.reset()  # Reset the LFSR state before descrambling
        descrambled_data = []
        for byte in data:
            descrambled_byte = byte ^ self.state
            descrambled_data.append(descrambled_byte)
            # Update the LFSR state
            self.state = self._update_state()
        return bytes(descrambled_data)

    def _update_state(self):
        """
        Update the LFSR state based on the polynomial.
        """
        new_bit = (self.state & 0x01)  # Extract the least significant bit
        self.state >>= 1  # Shift right
        if new_bit:
            self.state ^= self.polynomial  # XOR with the polynomial if the bit is set
        return self.state


# Example usage
if __name__ == "__main__":
    # Initialize the scrambler with a seed and polynomial
    scrambler = LFSRScrambler(seed=0xFF, polynomial=0x8A)

    # Example data to scramble
    original_data = b"Hello, world!"
    print(f"Original data: {original_data}")

    # Scramble the data
    scrambled_data = scrambler.scrambler(original_data)
    print(f"Scrambled data: {scrambled_data}")

    # Descramble the data
    descrambled_data = scrambler.descrambler(scrambled_data)
    print(f"Descrambled data: {descrambled_data}")
    assert original_data == descrambled_data


Original data: b'Hello, world!'
Scrambled data: b'\xb7\x90\x9c\x14S2/\xfa\xa3\x14_\xf7\xe2'
Descrambled data: b'Hello, world!'


In [11]:
class BitLevelLFSRScrambler:
    def __init__(self, seed=0xFF, polynomial=0x8A):
        """
        Initialize the bit-level LFSR scrambler.
        The polynomial 0x8A corresponds to x^8 + x^7 + x^5 + x^3 + 1.
        """
        self.seed = seed
        self.polynomial = polynomial
        self.reset()

    def reset(self):
        """Reset the LFSR state to the initial seed."""
        self.state = self.seed

    def scrambler(self, data):
        """
        Scramble the input data at the bit level.
        """
        scrambled_data = []
        for byte in data:
            scrambled_byte = 0
            for i in range(8):
                # Extract the current bit
                bit = (byte >> (7 - i)) & 0x01
                # Scramble the bit
                scrambled_bit = bit ^ ((self.state >> 7) & 0x01)
                scrambled_byte = (scrambled_byte << 1) | scrambled_bit
                # Update the LFSR state
                self.state = self._update_state()
            scrambled_data.append(scrambled_byte)
        return bytes(scrambled_data)

    def descrambler(self, data):
        """
        Descramble the input data at the bit level.
        """
        self.reset()  # Reset the LFSR state before descrambling
        descrambled_data = []
        for byte in data:
            descrambled_byte = 0
            for i in range(8):
                # Extract the current bit
                bit = (byte >> (7 - i)) & 0x01
                # Descramble the bit
                descrambled_bit = bit ^ ((self.state >> 7) & 0x01)
                descrambled_byte = (descrambled_byte << 1) | descrambled_bit
                # Update the LFSR state
                self.state = self._update_state()
            descrambled_data.append(descrambled_byte)
        return bytes(descrambled_data)

    def _update_state(self):
        """
        Update the LFSR state based on the polynomial.
        """
        new_bit = (self.state & 0x01)  # Extract the least significant bit
        self.state >>= 1  # Shift right
        if new_bit:
            self.state ^= self.polynomial  # XOR with the polynomial if the bit is set
        return self.state

# Example usage
if __name__ == "__main__":
    scrambler = BitLevelLFSRScrambler(seed=0xFF, polynomial=0x8A)
    original_data = b"Hello, world!"
    scrambled_data = scrambler.scrambler(original_data)
    descrambled_data = scrambler.descrambler(scrambled_data)

    print(f"Original data: {original_data}")
    print(f"Scrambled data: {scrambled_data}")
    print(f"Scrambled bitstream: {''.join([bin(byte)[2:] for byte in scrambled_data])}")
    print(f"Descrambled data: {descrambled_data}")


Original data: b'Hello, world!'
Scrambled data: b'\xa9\xfa\xea\x12v\xd4G\x96\xf0\xf4\x12}\xd9'
Scrambled bitstream: 10101001111110101110101010010111011011010100100011110010110111100001111010010010111110111011001
Descrambled data: b'Hello, world!'
